In [1]:
import pandas as pd
from tqdm import tqdm
import os
import yaml

import sys
sys.path.append('..')
from mlrose_hiive import FlipFlopGenerator
from mlrose_hiive import GARunner

In [2]:
SEED = 0
PROBLEM_SIZE_LIST = [yaml.load(open('parameters.yml'), yaml.Loader)['problem_size']]
ITERATIONS_LIST = [yaml.load(open('parameters.yml'), yaml.Loader)['iterations']]
MAX_ATTEMPTS_LIST = [yaml.load(open('parameters.yml'), yaml.Loader)['attempts']]
NUM_RUNS = yaml.load(open('parameters.yml'), yaml.Loader)['num_runs']
POPULATION_SIZES_LIST = [yaml.load(open('parameters.yml'), yaml.Loader)['population_size']]
MUTATION_RATE_LIST = [0.25, 0.5, 0.75, 0.9]

assert(len(PROBLEM_SIZE_LIST) == 1)
EXPERIMENT_NAME = f'FlipFlop_{PROBLEM_SIZE_LIST[0]}_iters={ITERATIONS_LIST[0]}_pop={POPULATION_SIZES_LIST[0]}_att={MAX_ATTEMPTS_LIST[0]}_GA'

In [3]:
output_dir = f'metrics/{EXPERIMENT_NAME}'
os.makedirs(output_dir, exist_ok=True)

In [4]:
all_df = pd.DataFrame()
group_i = 0
run_i = 0
for problem_size in PROBLEM_SIZE_LIST:
    print(f'Problem Size: {problem_size}')
    for iterations in ITERATIONS_LIST:
        print(f'Iterations: {iterations}')
        for max_attempts in MAX_ATTEMPTS_LIST:
            print(f'Max Attempts: {max_attempts}')
            for population_size in POPULATION_SIZES_LIST:
                print(f'Population Size: {population_size}')
                for mutation_rate in MUTATION_RATE_LIST:
                    print(f"Mutation Rate: {mutation_rate}")
                    for i in tqdm(range(NUM_RUNS)):
                        problem = FlipFlopGenerator().generate(seed=SEED+i, size=problem_size)
                        sa = GARunner(
                            problem=problem,
                            experiment_name='dontcare',
                            output_directory='./',
                            seed=SEED+i,
                            max_attempts=max_attempts,
                            iteration_list=[iterations],
                            population_sizes=[population_size],
                            mutation_rates=[mutation_rate]
                        )
                        _, df_run_curves = sa.run()
                        df_run_curves['group_number'] = group_i
                        df_run_curves['run_number'] = run_i
                        df_run_curves['problem_size'] = problem_size
                        df_run_curves['max_iterations'] = iterations
                        df_run_curves['max_attempts'] = max_attempts
                        df_run_curves['population_size'] = population_size
                        df_run_curves['mutation_rate'] = mutation_rate

                        print(f"Max Fitness: {df_run_curves['Fitness'].max()}")

                        all_df = pd.concat([all_df, df_run_curves], axis=0)
                        run_i += 1
                    group_i += 1
all_df.reset_index(inplace=True, drop=True)

Problem Size: 100
Iterations: 1000000.0
Max Attempts: 100
Population Size: 100
Mutation Rate: 0.25


 33%|███▎      | 1/3 [00:00<00:01,  1.20it/s]

Max Fitness: 89.0


 67%|██████▋   | 2/3 [00:02<00:01,  1.08s/it]

Max Fitness: 94.0


...

Max Fitness: 91.0


 67%|██████▋   | 2/3 [00:01<00:00,  1.30it/s]

Max Fitness: 91.0


100%|██████████| 3/3 [00:03<00:00,  1.06s/it]

Max Fitness: 94.0


In [5]:
print(f"Max: {all_df['Fitness'].max()}")
print(f"Min: {all_df['Fitness'].min()}")

Max: 95.0
Min: 38.0


In [6]:
all_df.to_csv(os.path.join(output_dir, 'learning_curve.csv'), index=False)